In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

source_table = 'citibike_lakehouse.gold.hourly_demand'

df = spark.table(source_table)

display(df)
df.printSchema()

In [0]:
ml_df = (
    df.select(
        "hour",
        "day_of_week",
        "trip_count"
    )
    .dropna()
)

display(ml_df)

In [0]:
assembler = VectorAssembler(
    inputCols=["hour", "day_of_week"],
    outputCol="features"
)

ml_read_df = assembler.transform(ml_df)

display(ml_read_df)

In [0]:
train_df, test_df = ml_read_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

In [0]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_count",
    predictionCol="prediction",
    numTrees=50,
    maxDepth=5,
    seed=42
)

model = rf.fit(train_df)

In [0]:
predictions = model.transform(test_df)

display(
    predictions.select(
        "hour",
        "day_of_week",
        "trip_count",
        "prediction"
    )
)

In [0]:
rmse_evaluator = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="mae"
)

rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS citibike_lakehouse.ml;

CREATE VOLUME IF NOT EXISTS citibike_lakehouse.ml.model_artifacts;

In [0]:
model_path = "/Volumes/citibike_lakehouse/ml/model_artifacts/bike_demand_random_forest"

model.write().overwrite().save(model_path)

print(f"Model saved to {model_path}")